# 04 · Multiple Tools

Real assistants have more than one tool. Here the model gets three:

| Tool | What it does |
|---|---|
| `get_weather` | Real weather from Open-Meteo |
| `get_today` | Today's date |
| `percentage_of` | A small calculator |

The model reads every description and **chooses** which tools fit the question. It can also ask for **several tools at once**.

> 💡 Why a `percentage_of` function and not a general "calculator" that runs any maths text? Tools should do **one small, safe thing**. Letting a model run arbitrary code is risky.

In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()   # read your keys from the .env file
assert os.getenv("OPENAI_API_KEY"), "Add OPENAI_API_KEY to your .env file (see .env.example)"

client = OpenAI()        # finds OPENAI_API_KEY on its own
MODEL = "gpt-4o-mini"    # a small, cheap model is plenty for this module
print("✅ Ready")

✅ Ready


In [2]:
import requests

def get_weather(city):
    """Get the current weather for a city using the free Open-Meteo API (no API key needed)."""

    # Step 1: turn the city name into latitude and longitude
    geo = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": city, "count": 1},
        timeout=10,
    ).json()
    if "results" not in geo:
        return f"Sorry, I could not find a city called {city}."
    place = geo["results"][0]

    # Step 2: ask for the weather at that location
    data = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": place["latitude"],
            "longitude": place["longitude"],
            "current": "temperature_2m,precipitation",
        },
        timeout=10,
    ).json()
    now = data["current"]

    return f"{place['name']}: {now['temperature_2m']}°C, rain right now: {now['precipitation']} mm"

In [3]:
from datetime import date

def get_today():
    """Return today's date. A model does not know what day it is, so this is a handy tool."""
    return date.today().isoformat()

def percentage_of(percent, amount):
    """Work out a percentage, for example 18% of 2400."""
    return percent / 100 * amount

# The descriptions below are what the model reads to decide WHICH tool to use,
# so write them clearly.
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather (temperature and rain) for a city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "City name, for example Pune"},
                },
                "required": ["city"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_today",
            "description": "Get today's date.",
            "parameters": {"type": "object", "properties": {}},   # this tool needs no input
        },
    },
    {
        "type": "function",
        "function": {
            "name": "percentage_of",
            "description": "Calculate a percentage of an amount, for example 18 percent of 2400.",
            "parameters": {
                "type": "object",
                "properties": {
                    "percent": {"type": "number", "description": "The percentage, for example 18"},
                    "amount": {"type": "number", "description": "The amount to take the percentage of"},
                },
                "required": ["percent", "amount"],
            },
        },
    },
]

# A lookup table: tool name -> the real Python function
available_tools = {
    "get_weather": get_weather,
    "get_today": get_today,
    "percentage_of": percentage_of,
}

def run_tool(name, args):
    """Run the tool the model asked for and return the result as text."""
    if name not in available_tools:
        return f"Unknown tool: {name}"
    try:
        return str(available_tools[name](**args))
    except Exception as error:      # if a tool breaks, tell the model instead of crashing
        return f"The tool failed: {error}"

## One question that needs all three tools

In [4]:
messages = [{"role": "user", "content": "Tell me today's date, the weather in Pune, and what 18% of 2400 is."}]

response = client.chat.completions.create(model=MODEL, messages=messages, tools=tools)
message = response.choices[0].message

print(f"The model asked for {len(message.tool_calls)} tools at once:")
for tool_call in message.tool_calls:
    print("  -", tool_call.function.name, tool_call.function.arguments)

The model asked for 3 tools at once:
  - get_today {}
  - get_weather {"city": "Pune"}
  - percentage_of {"percent": 18, "amount": 2400}


The model asked for all three in a single reply. We run each one and send back **one `tool` message per request**.

In [5]:
messages.append(message)    # the model's requests

for tool_call in message.tool_calls:
    args = json.loads(tool_call.function.arguments)
    result = run_tool(tool_call.function.name, args)
    print(f"{tool_call.function.name}({args}) -> {result}")

    messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": result})

final = client.chat.completions.create(model=MODEL, messages=messages, tools=tools)
print("\nFinal answer:\n", final.choices[0].message.content)

get_today({}) -> 2026-11-20


get_weather({'city': 'Pune'}) -> Pune: 18.6°C, rain right now: 0.0 mm
percentage_of({'percent': 18, 'amount': 2400}) -> 432.0



Final answer:
 Today's date is November 20, 2026. The weather in Pune is currently 18.6°C with no rain. Additionally, 18% of 2400 is 432.0.


## A question that needs no tool
With three tools available, the model still only uses one when it helps.

In [6]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "What is the capital of France?"}],
    tools=tools,
)
message = response.choices[0].message

print("Tool calls:", message.tool_calls)
print("Answer    :", message.content)

Tool calls: None
Answer    : The capital of France is Paris.


## What if a tool breaks?
`run_tool` catches errors and returns them as text. The model then reads the problem and can explain it, instead of your program crashing.

In [7]:
print(run_tool("percentage_of", {"percent": 18}))      # forgot the "amount" input
print(run_tool("book_a_flight", {"to": "Goa"}))        # a tool that does not exist

The tool failed: percentage_of() missing 1 required positional argument: 'amount'
Unknown tool: book_a_flight


## Recap
* Give the model a list of tools, each with a clear description.
* It chooses the tools, and can request several at once.
* Run each one, send back one `tool` message per request, then ask for the final answer.

Next: what if the model needs *more than one round* of tools? We build a small agent.